# Churn Prediction Model

This notebook contains the baseline churn prediction model for customer retention analysis.

## Overview
- Load and explore customer data
- Preprocess features
- Train baseline model
- Evaluate performance

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import yaml

In [ ]:
# Load configuration
with open('../configs/churn_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(config)

In [ ]:
# Import preprocessing functions
import sys
sys.path.append('..')
from preprocessing.clean_data import load_data, handle_missing_values, encode_categorical

## 1. Data Loading

Load the customer churn dataset.

In [ ]:
# TODO: Replace with actual data path
# df = load_data('path/to/churn_data.csv')

# For demonstration, create sample data
np.random.seed(config['model']['random_state'])
n_samples = 1000

df = pd.DataFrame({
    'customer_id': range(n_samples),
    'tenure_months': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 100, n_samples),
    'total_charges': np.random.uniform(100, 5000, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n_samples),
    'churn': np.random.choice([0, 1], n_samples, p=[0.73, 0.27])
})

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Data Preprocessing

In [ ]:
# Handle missing values
df_clean = handle_missing_values(df, strategy=config['preprocessing']['missing_value_strategy'])

# Encode categorical variables
categorical_cols = ['contract_type', 'payment_method']
df_encoded = encode_categorical(df_clean, categorical_cols)

print(f"Preprocessed dataset shape: {df_encoded.shape}")

## 3. Model Training

In [ ]:
# Prepare features and target
feature_cols = [col for col in df_encoded.columns if col not in ['customer_id', 'churn']]
X = df_encoded[feature_cols]
y = df_encoded['churn']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=config['model']['test_size'],
    random_state=config['model']['random_state']
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Train Random Forest model
model = RandomForestClassifier(
    n_estimators=config['model']['n_estimators'],
    max_depth=config['model']['max_depth'],
    random_state=config['model']['random_state']
)

model.fit(X_train, y_train)
print("Model training complete.")

## 4. Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top Feature Importances:")
feature_importance.head(10)

## Next Steps

- [ ] Experiment with hyperparameter tuning
- [ ] Try additional feature engineering
- [ ] Test alternative models (XGBoost, LightGBM)
- [ ] Implement cross-validation